In [23]:
# Goals: torch.nn module and torch.optim module

In [24]:
# !pip install torchinfo

In [2]:
import torch
import torch.nn as nn

## Simple neural network

In [3]:
class myModel(nn.Module): 6

    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        output = self.linear(features)
        output = self.sigmoid(output)
        return output

In [4]:
# create dataset
features = torch.rand(10,5)
features

tensor([[0.4506, 0.7314, 0.9316, 0.5504, 0.7123],
        [0.4341, 0.9632, 0.2510, 0.9162, 0.2430],
        [0.3408, 0.6225, 0.1220, 0.3101, 0.2208],
        [0.2488, 0.6538, 0.2859, 0.7956, 0.5410],
        [0.3512, 0.5216, 0.0702, 0.9850, 0.7147],
        [0.1136, 0.5901, 0.2428, 0.8472, 0.4583],
        [0.4488, 0.1415, 0.7103, 0.2888, 0.2996],
        [0.2363, 0.8210, 0.1723, 0.1833, 0.5980],
        [0.2286, 0.8833, 0.5246, 0.0150, 0.4052],
        [0.1355, 0.2101, 0.2470, 0.1688, 0.7523]])

In [8]:
# create model
model = myModel(features.shape[1])

# call forward pass on the model
model(features)  # this way is recommened if we are using pytorch
# model.forward(features)   # this is a traditional python oops way of calling a method

tensor([[0.5523],
        [0.5668],
        [0.4897],
        [0.5145],
        [0.5047],
        [0.5114],
        [0.5418],
        [0.4400],
        [0.4736],
        [0.4205]], grad_fn=<SigmoidBackward0>)

In [7]:
# read more about a python magic method __call__


In [9]:
# view model weights
model.linear.weight

Parameter containing:
tensor([[ 0.3359,  0.0182,  0.3713,  0.3838, -0.3675]], requires_grad=True)

In [10]:
model.linear.bias

Parameter containing:
tensor([-0.2501], requires_grad=True)

In [14]:
from torchinfo import summary
summary(model, input_size=(10,5), device='cpu')

Layer (type:depth-idx)                   Output Shape              Param #
myModel                                  [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   6
├─Sigmoid: 1-2                           [10, 1]                   --
Total params: 6
Trainable params: 6
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

## Neural Network with hidden leayer

In [16]:
class myModel(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.linear1 = nn.Linear(num_features, 3)   # num_features as input and 3 as output
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(3, 1)   # 3 inputs and 1 output
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear1(features)
        out = self.relu(out)
        out = self.linear2(out)
        out = self.sigmoid(out)
        return out

In [17]:
model = myModel(features.shape[1])

In [20]:
model(features)

tensor([[0.4349],
        [0.4352],
        [0.4398],
        [0.4347],
        [0.4330],
        [0.4331],
        [0.4300],
        [0.4464],
        [0.4456],
        [0.4392]], grad_fn=<SigmoidBackward0>)

In [21]:
model.linear1.weight
# weights of first layer

Parameter containing:
tensor([[-0.2341, -0.3895,  0.2240, -0.3169, -0.4164],
        [ 0.1836,  0.1113,  0.3774,  0.1700,  0.2796],
        [ 0.0622,  0.4452, -0.0161, -0.3078,  0.3369]], requires_grad=True)

In [23]:
model.linear2.weight

Parameter containing:
tensor([[ 0.2230, -0.0760,  0.1368]], requires_grad=True)

In [24]:
model.linear1.bias

Parameter containing:
tensor([ 0.0400, -0.3056,  0.1300], requires_grad=True)

In [25]:
summary(model, input_size=(10,5), device='cpu')

Layer (type:depth-idx)                   Output Shape              Param #
myModel                                  [10, 1]                   --
├─Linear: 1-1                            [10, 3]                   18
├─ReLU: 1-2                              [10, 3]                   --
├─Linear: 1-3                            [10, 1]                   4
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

## Sequential Containers

In [26]:
class myModel(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 3),   # num_features as input and 3 as output
            nn.ReLU(),
            nn.Linear(3, 1),   # 3 inputs and 1 output
            nn.Sigmoid(),
        )


    def forward(self, features):
        out = self.network(features)
        return out

In [27]:
model = myModel(features.shape[1])

In [28]:
model(features)

tensor([[0.5148],
        [0.5731],
        [0.5854],
        [0.5635],
        [0.5721],
        [0.5642],
        [0.5772],
        [0.5549],
        [0.5298],
        [0.5672]], grad_fn=<SigmoidBackward0>)

## Training pipeline in real projet

In [30]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [31]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [32]:
df.drop(columns=['Unnamed: 32', 'id'], inplace=True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [33]:
x_train, x_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

# Sacling
scalar = StandardScaler()
x_train = scalar.fit_transform(x_train)
x_test  = scalar.transform(x_test)

# Encoding
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# numpy arrays to pytorch tensors
x_train_tensor = torch.from_numpy(x_train)
x_test_tesnor = torch.from_numpy(x_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

#### Defining the model

In [35]:
class mySimpleNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 1),
            nn.Sigmoid(),
        )

    def forward(self, features):
        output = self.network(features)
        return output

    def loss_functon(self, y_pred, y):
        e = 1e-7
        prediction = torch.clamp(y_pred, e, 1-e)
        loss =  -(y*torch.log(prediction) + (1-y)*torch.log(1-prediction)).mean()

        return loss

In [36]:
learning_rate = 0.1
epochs = 25

In [45]:
# Setup model, loss, and optimizer
model = mySimpleNN(x_train_tensor.shape[1])
loss_function = nn.BCELoss()  # Binary Cross Entropy
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # Stochastic Gradient decent

# Ensure y_train_tensor matches y_pred shape (N, 1) and data type
y_train_tensor = y_train_tensor.view(-1, 1).float()  # for reshaping
x_train_tensor = x_train_tensor.float()

In [46]:
for epoch in range(epochs):
    # 1. Zero gradients from previous step
    optimizer.zero_grad()

    # 2. Forward pass
    y_pred = model(x_train_tensor)

    # 3. Calculate loss
    loss = loss_function(y_pred, y_train_tensor)

    # Clear Gradient
    optimizer.zero_grad()
    # 4. Backward pass
    loss.backward()

    # 5. Update weights automatically
    optimizer.step()

    print(f"Epoch {epoch+1} Loss: {loss.item():.4f}")

Epoch 1 Loss: 0.7723
Epoch 2 Loss: 0.5611
Epoch 3 Loss: 0.4530
Epoch 4 Loss: 0.3892
Epoch 5 Loss: 0.3471
Epoch 6 Loss: 0.3170
Epoch 7 Loss: 0.2942
Epoch 8 Loss: 0.2763
Epoch 9 Loss: 0.2618
Epoch 10 Loss: 0.2497
Epoch 11 Loss: 0.2393
Epoch 12 Loss: 0.2304
Epoch 13 Loss: 0.2226
Epoch 14 Loss: 0.2157
Epoch 15 Loss: 0.2095
Epoch 16 Loss: 0.2039
Epoch 17 Loss: 0.1988
Epoch 18 Loss: 0.1942
Epoch 19 Loss: 0.1899
Epoch 20 Loss: 0.1860
Epoch 21 Loss: 0.1824
Epoch 22 Loss: 0.1790
Epoch 23 Loss: 0.1758
Epoch 24 Loss: 0.1729
Epoch 25 Loss: 0.1701


In [47]:
# If using nn.Sequential inside model.network
weights = model.network[0].weight
biases = model.network[0].bias

print("Weights:\n", weights)
print("Biases:\n", biases)

Weights:
 Parameter containing:
tensor([[ 0.1960,  0.1847,  0.1584,  0.3393,  0.0386,  0.2612,  0.1827,  0.3281,
          0.1129, -0.0792,  0.2043,  0.0894,  0.1840, -0.0083, -0.0973,  0.2158,
         -0.0281, -0.0778, -0.0135, -0.0492,  0.3552,  0.1349,  0.3922,  0.2568,
          0.1865,  0.0505,  0.2134,  0.1513,  0.1728, -0.0378]],
       requires_grad=True)
Biases:
 Parameter containing:
tensor([-0.2572], requires_grad=True)


###  More Optimised Code

In [41]:
class mySimpleNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 1),
            nn.Sigmoid(),
        )

    def forward(self, features):
        output = self.network(features)
        return output


In [42]:
loss_function = nn.BCELoss()

In [43]:
loss_function

BCELoss()